# Tarea para el Hogar 05

##  1. Overfitting the Public Leaderboard

Leer  https://medium.com/hmif-itb/overfitting-the-leaderboard-da25172ac62e
( 8 minutos )

## 2. Hiperparámetros del LightGBM

Los objetivos de esta tarea son:


*   Aumentar la rentabilidad de la campaña de marketing de retención proactiva de clientes.
*   Generar un mejor modelo optimizando sus hiperparámetros
*   Conceptual : investigar los mas relevantes hiperparámetros de LightGBM
*   Familiarizarse con el uso de máquinas virtuales de Google Colab
*   Ver un pipeline completo de optimización de hiperparámetros y puesta en producción

LightGBM cuenta con mas de 60 hiperparámetros, siendo posible utilizar 40 al mismo tiempo, aunque no razonable.
<br> La documentación oficial de los hiperparámetros de LightGBM es  https://lightgbm.readthedocs.io/en/latest/Parameters.html#core-parameters


Se lo alerta sobre que una Optimizacion sw Hiperparámetros lleva varias horas de corrida, y usted deberá correr VARIAS optimizaciones para descubrir cuales parámetros conviene optimizar.


Es necesario investigar cuales son los hiperparámetros de LightGBM que vale la pena optimizar, ya que los realmente utiles son apenas un reducido subconjunto.
<br>Usted deberá investigar cuales son los hiperparámetros mas relevantes de LightGBM, su primer alternativa es preguntándole a su amigo con capacidades especiales ChatGPT o sus endogámicos familiares Claude, DeepSeek, Gemini, Grok, etc
<br> La segunda alternativa es la propia documentación de LightGBM  https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html


Adicionalmente podra buscar información como la que proveen esta diminuta muestra aleatoria de artículos ligeros:
* https://machinelearningmastery.com/light-gradient-boosted-machine-lightgbm-ensemble/
*  https://medium.com/@sarahzouinina/a-deep-dive-into-lightgbm-how-to-choose-and-tune-parameters-7c584945842e
*  https://www.kaggle.com/code/somang1418/tuning-hyperparameters-under-10-minutes-lgbm
*  https://towardsdatascience.com/beginners-guide-to-the-must-know-lightgbm-hyperparameters-a0005a812702/


<br>  La muestra anterior se brinda a modo de ejemplo, usted deberá buscar muuuuchas  fuentes adicionales de información
<br> Tenga presente que LightGBM es el estado del arte en modelado predictivo para datasets estructurado, que son el 90% del trabajo del 95% de los Data Scientists en Argentina.

El desafío de esta tarea es:
* Qué hiperparparámetros conviene optimizar?  Las recomendaciones de los artículos ligeros es siempre sensata?  Sus autores realmente hicieron experimentos o son siemplemente escritores de entretenimiento carente de base científica?
* Elegidos los hiperparámetros, cual es el  <desde, hasta> que se debe utilizar en la Bayesian Optimization ?
* Realmente vale la pena optimizar 10 o 16 hiperparámetros al mismo tiempo ?  No resulta contraproducente una búsqueda en un espacio de tal alta dimensionalidad ?

#### 2.1  Seteo del ambiente local


Esta parte se debe correr con un kernel de R local.
<br>En Jupyter, seleccionar el kernel **R** antes de ejecutar el notebook.


Los archivos persistentes quedan en el repo local: datasets en `datasets/` y resultados en `exp/`.


In [1]:
# Seteo local: resuelve el root del repo (funciona desde raiz, src/ensembles o exp/<exp>)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")
KAGGLE_JSON <- file.path(ROOT_DIR, "kaggle.json")


Para correr localmente, el dataset debe estar en `datasets/` dentro del repo.

<br>Si se va a subir a Kaggle, copiar `kaggle.json` a la raiz del repo antes de correr la siguiente celda. La celda lo instala en `~/.kaggle/kaggle.json` con permisos correctos.


In [2]:
dir.create(DATA_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)

dataset_local <- file.path(DATA_DIR, "dataset_pequeno.csv")
if (!file.exists(dataset_local)) {
  stop("No encuentro el dataset en: ", dataset_local)
}

if (file.exists(KAGGLE_JSON)) {
  kaggle_dir <- path.expand("~/.kaggle")
  dir.create(kaggle_dir, recursive = TRUE, showWarnings = FALSE)
  file.copy(KAGGLE_JSON, file.path(kaggle_dir, "kaggle.json"), overwrite = TRUE)
  Sys.chmod(file.path(kaggle_dir, "kaggle.json"), mode = "0600")
} else {
  message("No encontre kaggle.json en la raiz del repo. Solo es necesario para hacer submit a Kaggle.")
}


### 2.2 Optimizacion Hiperparámetros

Esta parte se corre en R (mismo kernel local). No hace falta cambiar de runtime de Colab.


### 2.2.1 Inicio

limpio el ambiente de R

In [3]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Aug 09 23:29:11 2026"

In [4]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,676829,36.2,1489770,79.6,NA,1489770,79.6
Vcells,1259247,9.7,8388608,64.0,49152,2014490,15.4


### 2.2.2 Carga de Librerias

In [5]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: parallel

Loading required package: primes

Loading required package: rlist

Loading required package: yaml

Loading required package: lightgbm

Warning message:
“package ‘lightgbm’ was built under R version 4.6.1”


### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [ ]:
PARAM <- list()
PARAM$experimento <- 5953

PARAM$semilla_primigenia <- 300089

In [7]:
PARAM$kaggle$competencia <- "data-mining-inicial-2026-b"
PARAM$kaggle$cortes <- seq(9000, 12000, by= 500)

In [8]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.5

In [9]:
# Parametros LightGBM

PARAM$hyperparametertuning$xval_folds <- 5

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "auc",
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE, # para reducir warnings
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L, # -1 significa no limitar,  por ahora lo dejo fijo
  min_gain_to_split= 0, # min_gain_to_split >= 0
  min_sum_hessian_in_leaf= 0.001, #  min_sum_hessian_in_leaf >= 0.0
  lambda_l1= 0.0, # lambda_l1 >= 0.0
  lambda_l2= 0.0, # lambda_l2 >= 0.0
  max_bin= 31L, # lo debo dejar fijo, no participa de la BO

  bagging_fraction= 1.0, # 0.0 < bagging_fraction <= 1.0
  pos_bagging_fraction= 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction= 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance= FALSE, #
  scale_pos_weight= 1.0, # scale_pos_weight > 0.0

  drop_rate= 0.1, # 0.0 < neg_bagging_fraction <= 1.0
  max_drop= 50, # <=0 means no limit
  skip_drop= 0.5, # 0.0 <= skip_drop <= 1.0

  extra_trees= FALSE,

  num_iterations= 100,
  learning_rate= 0.05,  # >=0
  feature_fraction= 1.0, # 0 < ff <= 1.0
  num_leaves= 20, # integer >= 2
  min_data_in_leaf= 20, # integer >= 0


  # sampling
  data_sample_strategy = "goss",
  top_rate = 0.4
)


### 2.2.4  Preprocesamiento

In [10]:
# carpeta de trabajo (re-resuelve paths por si hubo rm() arriba)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")

dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)
experimento_folder <- paste0("HT", PARAM$experimento)
dir.create(file.path(EXP_DIR, experimento_folder), recursive = TRUE, showWarnings = FALSE)
setwd(file.path(EXP_DIR, experimento_folder))


In [11]:
# lectura del dataset
dataset <- fread(file.path(DATA_DIR, "dataset_pequeno.csv"))


In [12]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [13]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0

dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [14]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA
# notar que para esto utilizo la SEGUNDA semilla

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]

dataset_train[
  foto_mes %in% c(202107) &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

In [15]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "azar", "training")
)

In [16]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

[1] 83370

[1] 154

2.2.5 Configuracion del Grid Search

In [17]:
# En el argumento x llegan los parmaetros de LightGBM
#  devuelve la AUC en cross validation del modelo entrenado

Estimar_AUC_lightgbm <- function(x) { # nolint

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelocv <- lgb.cv(
    data= dtrain,
    nfold= PARAM$hyperparametertuning$xval_folds,
    stratified= TRUE,
    param= param_completo
  )

  # obtengo la ganancia
  AUC <- modelocv$best_score

  # hago espacio en la memoria
  rm(modelocv)
  gc(full= TRUE, verbose= FALSE)

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " AUC ", AUC
  )

  return(AUC)
}

In [18]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_iterations= c(1000, 1500, 2000),
  learning_rate= c( 0.02, 0.05, 0.1),
  num_leaves= c(10, 50, 100, 500)
)

In [19]:
# veo que tiene la tabla antes de procesar
tb_nueva

num_iterations,learning_rate,num_leaves
<dbl>,<dbl>,<dbl>
1000,0.02,10
1000,0.02,50
1000,0.02,100
1000,0.02,500
1000,0.05,10
1000,0.05,50
1000,0.05,100
1000,0.05,500
1000,0.10,10


In [20]:
# registro a registro calculo la AUC
tb_nueva[, AUC:= Estimar_AUC_lightgbm( list(.SD)),
  by=1:nrow(tb_nueva) ]

Sun Aug 09 23:29:18 2026  list(num_iterations = 1000, learning_rate = 0.02, num_leaves = 10) AUC 0.926436369252005

Sun Aug 09 23:29:22 2026  list(num_iterations = 1000, learning_rate = 0.02, num_leaves = 50) AUC 0.9281027989536

Sun Aug 09 23:29:27 2026  list(num_iterations = 1000, learning_rate = 0.02, num_leaves = 100) AUC 0.927375317516403

Sun Aug 09 23:29:32 2026  list(num_iterations = 1000, learning_rate = 0.02, num_leaves = 500) AUC 0.926821051141495

Sun Aug 09 23:29:37 2026  list(num_iterations = 1000, learning_rate = 0.05, num_leaves = 10) AUC 0.927272712705921

Sun Aug 09 23:29:42 2026  list(num_iterations = 1000, learning_rate = 0.05, num_leaves = 50) AUC 0.928431077759346

Sun Aug 09 23:29:47 2026  list(num_iterations = 1000, learning_rate = 0.05, num_leaves = 100) AUC 0.926615291860543

Sun Aug 09 23:29:51 2026  list(num_iterations = 1000, learning_rate = 0.05, num_leaves = 500) AUC 0.928721539729733

Sun Aug 09 23:29:56 2026  list(num_iterations = 1000, learning_rate = 

In [21]:
# veo que tiene la tabla DESPUES de procesar
tb_nueva

num_iterations,learning_rate,num_leaves,AUC
<dbl>,<dbl>,<dbl>,<dbl>
1000,0.02,10,0.9264364
1000,0.02,50,0.9281028
1000,0.02,100,0.9273753
1000,0.02,500,0.9268211
1000,0.05,10,0.9272727
1000,0.05,50,0.9284311
1000,0.05,100,0.9266153
1000,0.05,500,0.9287215
1000,0.10,10,0.9280078


In [22]:
fwrite( tb_nueva,
  file= "tb_grid_serach_01.txt",
  sep="\t",
  append= TRUE
)

In [23]:
setorder( tb_nueva, -AUC)
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC]
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

$num_iterations
[1] 1000

$learning_rate
[1] 0.05

$num_leaves
[1] 500

In [24]:
write_yaml( PARAM, file="PARAM.yml")

## 2.3  Produccion

### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [25]:
# carpeta de trabajo produccion (re-resuelve paths por si se cambio el wd)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")

dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)
experimento <- paste0("exp", PARAM$experimento)
dir.create(file.path(EXP_DIR, experimento), recursive = TRUE, showWarnings = FALSE)
setwd(file.path(EXP_DIR, experimento))


#### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización de hiperparametros

In [26]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [27]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [28]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)

#### Final Training Hyperparameters

In [29]:
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] FALSE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$force_row_wise
[1] TRUE

$verbosity
[1] -100

$seed
[1] 300089

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$min_sum_hessian_in_leaf
[1] 0.001

$lambda_l1
[1] 0

$lambda_l2
[1] 0

$max_bin
[1] 31

$bagging_fraction
[1] 1

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$num_iterations
[1] 1000

$learning_rate
[1] 0.05

$feature_fraction
[1] 1

$num_leaves
[1] 500

$min_data_in_leaf
[1] 20

$data_sample_strategy
[1] "goss"

$top_rate
[1] 0.4

#### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [30]:
# este punto es muy SUTIL  y será revisado en la Clase 05

param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <-  round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)

In [31]:
  # entreno LightGBM

  modelo_final <- lgb.train(
    data= dtrain,
    param= param_normalizado
  )

In [32]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(modelo_final))
archivo_importancia <- "impo.txt"

fwrite(tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

In [33]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(modelo_final, "modelo.txt" )

### Scoring

Aplico el modelo final a los datos del futuro

In [34]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes == 202109]

# aplico el modelo a los datos nuevos
prediccion <- predict(
  modelo_final,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

#### Tabla Prediccion

In [35]:
# tabla de prediccion

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion ]

# grabo las probabilidad del modelo
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

Kaggle Competition Submit

In [36]:
PARAM$kaggle$cortes

[1]  9000  9500 10000 10500 11000 11500 12000

In [37]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
  Sys.sleep(45)
}

92 submissions remaining today. Successfully submitted to Data Mining, Inicial 2026 B 
91 submissions remaining today. Successfully submitted to Data Mining, Inicial 2026 B 
90 submissions remaining today. Successfully submitted to Data Mining, Inicial 2026 B 
89 submissions remaining today. Successfully submitted to Data Mining, Inicial 2026 B 
88 submissions remaining today. Successfully submitted to Data Mining, Inicial 2026 B 
87 submissions remaining today. Successfully submitted to Data Mining, Inicial 2026 B 
86 submissions remaining today. Successfully submitted to Data Mining, Inicial 2026 B 


In [38]:
write_yaml( PARAM, file="PARAM.yml")

In [39]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Aug 09 23:39:37 2026"

Finalmente usted deberá cargar el resultado de su corrida en la Google Sheet Colaborativa,  hoja **TareaHogar-04**
<br> Siéntase libre de agregar las columnas que hagan falta a la planilla

Seguramente usted realice varias corridas de este script con distintos conjuntos de hiperparámetros, siempre cambiandole el nombre al script  y también cambiando el nombre del experimento,  deberá TODAS esas corridas en distintas lineas de la  Google Sheet Colaborativa, hoja **TareaHogar-04**

Siéntase libre de agregar columnas a la hoja **TareaHogar-04**  en caso de ser necesario.